# ComfyUI + AnimateDiff Setup

This notebook sets up ComfyUI with AnimateDiff for AI video generation.

**Requirements:**
- NVIDIA GPU with CUDA support (recommended)
- 10GB+ free disk space
- Internet connection

**Note:** This is adapted from Google Colab. Some features may work differently in local Jupyter.

## 1. Check GPU Availability

In [ ]:
# Check for NVIDIA GPU
!nvidia-smi

In [ ]:
import os
import torch

# Check PyTorch CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("? GPU ????! (GPU Available!)")
else:
    print("?? No GPU detected. ComfyUI will run on CPU (very slow)")

In [ ]:
# Set up project directory
HOME = os.path.expanduser("~")
PROJECT_DIR = f"{HOME}/comfyui-animatediff"
print(f"Project directory: {PROJECT_DIR}")

## 2. Clone ComfyUI Repository

In [ ]:
# Change to working directory
%cd ~

# Clone ComfyUI from GitHub
!git clone https://github.com/comfyanonymous/ComfyUI

## 3. Install Dependencies

In [ ]:
# Change to ComfyUI directory and install requirements
%cd ~/ComfyUI
!pip install -r requirements.txt

In [ ]:
# Install additional dependencies
!pip install xformers  # For faster attention mechanisms
!pip install ipywidgets  # For interactive widgets

## 4. Create Model Directories

In [ ]:
import os

# Define model directories
checkpoints_dir = os.path.expanduser("~/ComfyUI/models/checkpoints")
vae_dir = os.path.expanduser("~/ComfyUI/models/vae")
animatediff_dir = os.path.expanduser("~/ComfyUI/models/animatediff_models")

# Create directories
os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)
os.makedirs(animatediff_dir, exist_ok=True)

print("? Model directories created:")
print(f"  Checkpoints: {checkpoints_dir}")
print(f"  VAE: {vae_dir}")
print(f"  AnimateDiff: {animatediff_dir}")

## 5. Download Models

This will download:
- **Stable Diffusion v1.5** (~4GB)
- **VAE model** (~335MB)
- **AnimateDiff model** (~1.8GB)

**Note:** This may take 10-30 minutes depending on your internet speed.

In [ ]:
# Download Stable Diffusion v1.5
print("Downloading Stable Diffusion v1.5...")
!wget -P {checkpoints_dir} https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
print("? Stable Diffusion v1.5 downloaded")

In [ ]:
# Download VAE model
print("Downloading VAE model...")
!wget -P {vae_dir} https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors
print("? VAE model downloaded")

In [ ]:
# Download AnimateDiff model
print("Downloading AnimateDiff model...")
!wget -P {animatediff_dir} https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt
print("? AnimateDiff model downloaded")

In [ ]:
# Verify downloads
print("\n=== Downloaded Models ===")
print(f"\nCheckpoints ({checkpoints_dir}):")
!ls -lh {checkpoints_dir}

print(f"\nVAE ({vae_dir}):")
!ls -lh {vae_dir}

print(f"\nAnimateDiff ({animatediff_dir}):")
!ls -lh {animatediff_dir}

## 6. Start ComfyUI Server

**Important:** 
- This cell will start the ComfyUI server
- In Jupyter, you'll need to stop the cell (interrupt kernel) to continue
- Better to run ComfyUI from terminal: `python main.py --listen`

**Access the UI at:** http://localhost:8188

In [ ]:
# Note: This cell will run indefinitely. Use kernel interrupt to stop.
%cd ~/ComfyUI

# Start ComfyUI with external listening enabled
!python main.py --listen

## Alternative: Run ComfyUI in Background (Advanced)

For Jupyter, it's better to run ComfyUI from a terminal in the background.

In [ ]:
# Create a startup script
startup_script = os.path.expanduser("~/start_comfyui.sh")

with open(startup_script, 'w') as f:
    f.write('''#!/bin/bash
cd ~/ComfyUI
python main.py --listen &
echo "ComfyUI started in background"
echo "Access at: http://localhost:8188"
echo "To stop: pkill -f 'python main.py'"
''')

# Make it executable
!chmod +x {startup_script}

print(f"? Startup script created at: {startup_script}")
print("\nTo start ComfyUI in background, run in terminal:")
print(f"  {startup_script}")

## 7. Interactive Widget Interface (Optional)

This creates an interactive interface similar to Google Colab's AI features.

**Note:** This is a simplified version. Full Colab AI features require Google Colab environment.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

# Create widgets
text_input = widgets.Textarea(
    placeholder='Enter your prompt for image generation...',
    layout={'width': '100%', 'height': '100px'},
    description='Prompt:'
)

button = widgets.Button(
    description='Generate Info',
    disabled=False,
    tooltip='Click to get information',
    icon='check',
    button_style='success'
)

output_area = widgets.Output(
    layout={'width': '100%', 'max_height': '300px', 'overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        prompt = text_input.value
        
        if not prompt:
            print("Please enter a prompt!")
            return
        
        # Generate some helpful information
        info = f"""
### Prompt Analysis

**Your Prompt:** {prompt}

**Suggestions for ComfyUI:**
1. Load the Stable Diffusion checkpoint
2. Use a CLIP Text Encode node with this prompt
3. For animations, add AnimateDiff Loader
4. Set frame count (16-32 recommended)
5. Connect to KSampler for generation

**Recommended Settings:**
- Steps: 20-30
- CFG Scale: 7-9
- Sampler: euler or dpm++
- Resolution: 512x512 (for SD 1.5)

**Tips:**
- Be specific in your descriptions
- Use negative prompts to avoid unwanted elements
- Experiment with different seeds
        """
        
        display(Markdown(info))

button.on_click(on_button_clicked)

# Display widgets
display(HTML("<h3>ComfyUI Prompt Helper</h3>"))
display(widgets.VBox([text_input, button, output_area]))

## 8. Quick Status Check

In [ ]:
import requests

def check_comfyui_status():
    """Check if ComfyUI server is running"""
    try:
        response = requests.get("http://localhost:8188/system_stats", timeout=2)
        if response.status_code == 200:
            print("? ComfyUI server is running!")
            print("  Access at: http://localhost:8188")
            stats = response.json()
            if 'devices' in stats:
                for device in stats['devices']:
                    print(f"  GPU: {device.get('name', 'Unknown')}")
            return True
        else:
            print("?? Server responded but with unexpected status")
            return False
    except requests.exceptions.RequestException:
        print("? ComfyUI server is not running")
        print("\nTo start it:")
        print("  1. Open a terminal")
        print("  2. Run: cd ~/ComfyUI && python main.py --listen")
        return False

check_comfyui_status()

## 9. Useful Commands

Run these commands in your terminal:

In [ ]:
print("""
=== Useful Commands ==="

Start ComfyUI:
  cd ~/ComfyUI
  python main.py --listen

Start with low VRAM mode:
  python main.py --listen --lowvram

Start on different port:
  python main.py --listen --port 8080

Stop ComfyUI:
  Press Ctrl+C in the terminal
  Or: pkill -f 'python main.py'

Check GPU usage:
  watch -n 1 nvidia-smi

View output files:
  ls -lh ~/ComfyUI/output/

Clean output folder:
  rm ~/ComfyUI/output/*
""")

## 10. Next Steps

1. **Start ComfyUI** from terminal (see commands above)
2. **Open browser** to http://localhost:8188
3. **Load example workflows** from the ComfyUI interface
4. **Experiment** with different prompts and settings
5. **Try AnimateDiff** nodes for video generation

### Resources
- [ComfyUI Wiki](https://github.com/comfyanonymous/ComfyUI/wiki)
- [Example Workflows](https://comfyanonymous.github.io/ComfyUI_examples/)
- [Community Workflows](https://openart.ai/workflows)
- [r/comfyui](https://reddit.com/r/comfyui)

### Troubleshooting
- **Out of memory**: Use `--lowvram` flag
- **Slow generation**: Check GPU is being used
- **Models not found**: Verify files in models directory
- **Port in use**: Change port with `--port` flag

Happy creating! ???